# Lab 6 — Solución docente (Agentes + Ollama)

Referencia con tools, agente Agno + **Ollama local** e informe LaTeX.
Las celdas siguientes repiten la **guía de inicio** del notebook alumno (qué haremos + esquema).


## Qué haremos en este laboratorio

**Objetivo:** construir un **agente inspector estructural autónomo** que, ante una consulta en lenguaje natural, **orquesta herramientas especializadas** (no calcula por sí mismo) y entrega un **informe técnico en LaTeX**.

### Roadmap del notebook (10 secciones)

| # | Qué implementarás | Resultado esperado |
|---|-------------------|-------------------|
| 1 | Concepto **ReAct** (chat vs agente) | Lista `PASOS_REACT` |
| 2 | Verificar **Ollama** + best model `.pkl` | `OLLAMA_OK`, `PKL_OK` |
| 3 | **Tool 1** — telemetría del edificio | `get_building_telemetry` |
| 4 | **Tool 2** — norma ASCE 7 (drift) | `check_seismic_code_compliance` |
| 5 | **Tool 3** — riesgo sísmico (MLP) | `predict_earthquake_vulnerability` |
| 6 | **Agente** Agno + Ollama con 4 tools | `structural_bot` |
| 7 | **Ejecutar** agente sobre **BLDG-B** | Reporte + traza ReAct |
| 8 | **Comparar** BLDG-A / B / C | Tabla norma vs ML |
| 9 | **Tool 4** — export LaTeX | `informe_sismico.tex` |
| 10 | Puente hacia **MCP** + bitácora | `prompts_entregados.md` |

**Edificio principal del caso:** `BLDG-B` (concreto, suelo blando) — falla norma de drift y dispara alerta del MLP.

**Entregables:** notebook con ✅ · `informe_sismico.tex` · bitácora de prompts.


## Esquema del agente y las herramientas

### 1. El agente — `structural_bot`

| Componente | Tecnología | Función |
|------------|------------|---------|
| **Cerebro** | Ollama (`llama3.2:3b`) | Razona el siguiente paso y redacta el dictamen |
| **Orquestador** | Agno (`Agent`) | Bucle **ReAct**: observar → pensar → llamar tool → verificar |
| **Nombre** | `structural_bot` | Agente inspector sísmico del laboratorio |

```mermaid
flowchart LR
  subgraph agentBox [Agente structural_bot]
    Ollama[Ollama LLM]
    React[ReAct loop]
  end
  User[Usuario] -->|consulta BLDG-B| React
  React <--> Ollama
  React --> T1
  React --> T2
  React --> T3
  React --> T4
  subgraph toolsBox [Tools registradas en Agno]
    T1[get_building_telemetry]
    T2[check_seismic_code_compliance]
    T3[predict_earthquake_vulnerability]
    T4[export_latex_report]
  end
  T4 --> Tex[informe_sismico.tex]
```

### 2. Las cuatro herramientas (tools)

| # | Función Python | Entrada principal | Salida | Naturaleza |
|---|----------------|-------------------|--------|------------|
| 1 | `get_building_telemetry(building_id)` | ID (`BLDG-A/B/C`) | `dict` con drift, material, PGA, periodo, suelo | **Determinista** — consulta `TELEMETRIA_DB` |
| 2 | `check_seismic_code_compliance(max_drift_ratio, material)` | Drift medido + material | `FALLO` o `APROBADO` (ASCE 7) | **Determinista** — regla fija (2% concreto, 2.5% acero) |
| 3 | `predict_earthquake_vulnerability(pga, suelo, periodo, …)` | Variables sísmico-estructurales | Mensaje + probabilidad de colapso | **Probabilística** — MLP `.pkl` pre-entrenado |
| 4 | `export_latex_report(building_id, dictamen, …)` | Resultados de tools 1–3 + dictamen LLM | Ruta `informe_sismico.tex` | **Determinista** — plantilla LaTeX trazable |

### 3. Flujo típico al evaluar un edificio

```
Usuario: «Inspecciona BLDG-B y genera informe»
    │
    ▼
Agente (Ollama) — piensa: «Necesito datos del edificio»
    │
    ├─► [1] get_building_telemetry("BLDG-B")  → drift=3.6%, concreto, suelo 3
    │
    ├─► [2] check_seismic_code_compliance(0.036, "Concreto…")  → FALLO norma
    │
    ├─► [3] predict_earthquake_vulnerability(…)  → ALERTA ML ~95%
    │
    ├─► Ollama redacta dictamen (encamisado, muros de corte…)
    │
    └─► [4] export_latex_report(…)  → informe_sismico.tex
```

> **Idea clave:** el LLM **no sustituye** al ingeniero ni calcula drift; **decide qué tool invocar** y **sintetiza** resultados en lenguaje técnico.


## Caja de herramientas del Lab 6

> Construir un **agente inspector** que coordina telemetría, norma ASCE 7, un **MLP pre-entrenado** y exportación LaTeX — el **LLM (Ollama) no inventa cálculos**, solo orquesta tools.

| Herramienta | Rol | Analogía en obra |
|-------------|-----|------------------|
| **Ollama** | LLM local (`llama3.2:3b`) — razonamiento y dictamen | Ingeniero senior que redacta informes |
| **Agno** | Bucle ReAct + tool calling | Coordinador de oficina técnica |
| **MLP (.pkl)** | Red neuronal entrenada offline | Modelo especialista del laboratorio |
| **joblib** | Carga del best model | Archivo `.pkl` listo para producción |
| **LaTeX** | Plantilla `informe_sismico.tex` | Memoria de inspección trazable |

**Flujo:** Usuario → Agente (Ollama) → {telemetría, norma, MLP, LaTeX} → informe `.tex`


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import requests

sys.path.insert(0, str(Path('.').resolve()))
from _verificar import (
    TELEMETRIA_DB, RUTA_CSV, RUTA_PKL,
    evaluar_drift_asce7, predecir_probabilidad_dano, generar_latex_report,
    leer_meta_modelo, cargar_modelo_sismico,
    verificar_pasos_react, verificar_entorno, verificar_ollama,
    verificar_telemetria_b, verificar_resultado_norma_b, verificar_ml_b,
    verificar_agente_config, verificar_respuesta_agente,
    verificar_tabla_comparativa, verificar_latex_report, verificar, resumen_seccion,
)

MODELO_LLM = 'llama3.2:3b'
OLLAMA_URL = 'http://localhost:11434'
TRAZA_HERRAMIENTAS: list[str] = []

print('✅ Entorno listo — LLM: Ollama', MODELO_LLM, '@', OLLAMA_URL)
print('   Edificios demo:', list(TELEMETRIA_DB.keys()))


In [ ]:
# --- PRE-ESCRITO: conoce tu best model (no re-entrenar) ---
import subprocess
subprocess.run([sys.executable, '_preparar_datos.py'], check=False)

meta = leer_meta_modelo()
print('Tipo:', meta.get('tipo'), '| Arquitectura:', meta.get('architecture'))
print('test_r2:', meta.get('test_r2'), '| Features:', meta.get('features'))
_ = cargar_modelo_sismico()
print('✅ Best model cargado desde', RUTA_PKL)


## Dataset — Escenarios sísmico-estructurales

| ID demo | Narrativa |
|---------|-----------|
| **BLDG-A** | Acero / suelo duro — norma OK + ML bajo |
| **BLDG-B** | Concreto / suelo blando — **falla norma** + alerta ML |
| **BLDG-C** | Composite — **norma OK** pero **ML crítico** |

Detalle: [`data/DATOS.md`](data/DATOS.md).


## Pregunta 1 — Chat vs agente (ReAct)

### 📘 Subpreguntas
- **1.a** ¿Por qué un LLM solo no puede abrir tu base de datos de sensores?
- **1.b** ¿Qué aporta el bucle ReAct frente a un prompt único?


#### ✅ Respuesta sugerida

**1.a** El LLM no tiene acceso directo a sistemas externos; necesita **tools** acotadas.
**1.b** ReAct permite **observar → razonar → actuar → verificar** iterativamente.


In [ ]:
PASOS_REACT = ["Observar objetivo", "Razonar siguiente paso", "Invocar herramienta", "Verificar y sintetizar"]
print("Bucle ReAct:")
for i, paso in enumerate(PASOS_REACT, 1):
    print(f"  {i}. {paso}")


In [ ]:
# --- Autoevaluación 1 ---
# Requiere (celda «Aquí coloca tu solución»): `PASOS_REACT`
r = []
try:
    r = verificar_pasos_react(PASOS_REACT)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('1 — ReAct', r)


## Pregunta 2 — Ollama + best model MLP

### 📘 Subpreguntas
- **2.a** ¿Ventaja de Ollama en localhost:11434?
- **2.b** ¿Por qué no re-entrenamos el MLP en clase?


#### ✅ Respuesta sugerida

**2.a** Datos de obra no salen del contenedor; sin costo de tokens en la nube.
**2.b** El foco del lab es **orquestación del agente**, no entrenamiento ML.


In [ ]:
# --- PRE-ESCRITO: diagnóstico Ollama ---
# Si OLLAMA_OK es False: bash labs/lab6/_ollama_setup.sh
print("Comprueba Ollama antes de las secciones 6–7.")


In [ ]:
try:
    resp = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    OLLAMA_OK = resp.status_code == 200
except Exception:
    OLLAMA_OK = False
PKL_OK = RUTA_PKL.is_file()
N_FILAS_CSV = len(pd.read_csv(RUTA_CSV)) if RUTA_CSV.is_file() else 0
TIPO_MODELO = leer_meta_modelo().get('tipo', '?')
print(f"OLLAMA_OK={OLLAMA_OK} | PKL_OK={PKL_OK} | N_FILAS_CSV={N_FILAS_CSV} | TIPO_MODELO={TIPO_MODELO}")


In [ ]:
# --- Autoevaluación 2 ---
# Requiere (celda «Aquí coloca tu solución»): `MODELO_LLM`, `OLLAMA_OK`, `PKL_OK`, `N_FILAS_CSV`, `TIPO_MODELO`
r = []
try:
    r = verificar_entorno(OLLAMA_OK, PKL_OK, N_FILAS_CSV, TIPO_MODELO)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('2 — Ollama + MLP', r)


## Pregunta 3 — Tool: telemetría

### 📘 Subpreguntas
- **3.a** ¿Qué campos necesitas para la norma y el ML?


#### ✅ Respuesta sugerida

**3.a** `max_drift_ratio`, `material`, `expected_pga_g`, `soil_type_index`, `structural_period_s`.


In [ ]:
# --- PRE-ESCRITO: edificios disponibles ---
print(list(TELEMETRIA_DB.keys()))


In [ ]:
def get_building_telemetry(building_id: str) -> dict:
    TRAZA_HERRAMIENTAS.append(f'get_building_telemetry({building_id})')
    return TELEMETRIA_DB.get(building_id, {"error": f"ID {building_id} no existe."})

TELEMETRIA_B = get_building_telemetry('BLDG-B')
print(TELEMETRIA_B['name'], '| drift=', TELEMETRIA_B['max_drift_ratio'])


In [ ]:
# --- Autoevaluación 3 ---
# Requiere (celda «Aquí coloca tu solución»): `TELEMETRIA_B`
r = []
try:
    r = verificar_telemetria_b(TELEMETRIA_B)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('3 — Telemetría', r)


## Pregunta 4 — Tool: norma ASCE 7

### 📘 Subpreguntas
- **4.a** ¿Por qué BLDG-B falla si el límite de concreto es 2%?


#### ✅ Respuesta sugerida

**4.a** Su drift (~3.6%) supera el 2% permitido para concreto.


In [ ]:
def check_seismic_code_compliance(max_drift_ratio: float, material: str) -> str:
    TRAZA_HERRAMIENTAS.append('check_seismic_code_compliance(...)')
    return evaluar_drift_asce7(max_drift_ratio, material)

RESULTADO_NORMA_B = check_seismic_code_compliance(TELEMETRIA_B["max_drift_ratio"], TELEMETRIA_B["material"])
print(RESULTADO_NORMA_B)


In [ ]:
# --- Autoevaluación 4 ---
# Requiere (celda «Aquí coloca tu solución»): `RESULTADO_NORMA_B`
r = []
try:
    r = verificar_resultado_norma_b(RESULTADO_NORMA_B)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('4 — Norma', r)


## Pregunta 5 — Tool: MLP (.pkl)

### 📘 Subpreguntas
- **5.a** ¿Diferencia entre fallo normativo y alerta ML?


#### ✅ Respuesta sugerida

**5.a** La norma es **determinista** (regla fija); el MLP es **probabilístico** (datos + incertidumbre).


In [ ]:
def predict_earthquake_vulnerability(
    expected_pga_g: float, soil_type_index: int, structural_period_s: float,
    max_drift_ratio: float = 0.0, height_m: float = 10.0, material: str = 'Concrete',
) -> str:
    TRAZA_HERRAMIENTAS.append('predict_earthquake_vulnerability(...)')
    prob, msg = predecir_probabilidad_dano(
        expected_pga_g, soil_type_index, structural_period_s,
        max_drift_ratio=max_drift_ratio, height_m=height_m, material=material,
    )
    global PROB_RIESGO_B, MSG_ML_B
    PROB_RIESGO_B, MSG_ML_B = prob, msg
    return msg

predict_earthquake_vulnerability(
    TELEMETRIA_B['expected_pga_g'], TELEMETRIA_B['soil_type_index'],
    TELEMETRIA_B['structural_period_s'], TELEMETRIA_B['max_drift_ratio'],
    TELEMETRIA_B['height_m'], TELEMETRIA_B['material'],
)
print(f'PROB_RIESGO_B={PROB_RIESGO_B:.3f}')
print(MSG_ML_B)


In [ ]:
# --- Autoevaluación 5 ---
# Requiere (celda «Aquí coloca tu solución»): `PROB_RIESGO_B`, `MSG_ML_B`
r = []
try:
    r = verificar_ml_b(PROB_RIESGO_B, MSG_ML_B)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('5 — MLP', r)


## Pregunta 6 — Agente Agno + Ollama

### 📘 Subpreguntas
- **6.a** ¿Por qué pasamos funciones Python como `tools`?
- **6.b** ¿Qué hace `show_tool_calls=True`?


#### ✅ Respuesta sugerida

**6.a** El LLM elige qué ejecutar; las funciones encapsulan lógica verificable.
**6.b** Muestra el bucle ReAct en consola (tool calls visibles).


In [ ]:
# --- PRE-ESCRITO: import Agno + Ollama ---
from agno.agent import Agent
from agno.models.ollama import Ollama
print("Agno listo — modelo Ollama:", MODELO_LLM)


In [ ]:
def export_latex_report(
    building_id: str,
    dictamen_markdown: str,
    ruta_salida: str = 'informe_sismico.tex',
) -> str:
    TRAZA_HERRAMIENTAS.append('export_latex_report(...)')
    t = get_building_telemetry(building_id)
    if 'error' in t:
        return t['error']
    norma = check_seismic_code_compliance(t['max_drift_ratio'], t['material'])
    msg = predict_earthquake_vulnerability(
        t['expected_pga_g'], t['soil_type_index'], t['structural_period_s'],
        t['max_drift_ratio'], t['height_m'], t['material'],
    )
    prob = PROB_RIESGO_B
    path = generar_latex_report(
        building_id, t, norma, msg, prob, TRAZA_HERRAMIENTAS.copy(),
        dictamen_markdown, ruta_salida=ruta_salida,
    )
    return f'Informe LaTeX: {path}'

structural_bot = Agent(
    model=Ollama(id=MODELO_LLM),
    tools=[get_building_telemetry, check_seismic_code_compliance,
           predict_earthquake_vulnerability, export_latex_report],
    instructions=[
        'Eres un agente experto en ingeniería estructural sísmica.',
        'Para inspeccionar un edificio: 1) get_building_telemetry 2) check_seismic_code_compliance',
        '3) predict_earthquake_vulnerability 4) redacta dictamen 5) export_latex_report.',
        'Usa los valores de telemetría como argumentos exactos.',
    ],
    show_tool_calls=True,
    markdown=True,
)
N_TOOLS = 4
print('Agente configurado con Ollama +', N_TOOLS, 'tools')


In [ ]:
# --- Autoevaluación 6 ---
# Requiere (celda «Aquí coloca tu solución»): `N_TOOLS`
r = []
try:
    r = verificar_agente_config(structural_bot, N_TOOLS)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('6 — Agente', r)


## Pregunta 7 — Ejecutar agente (BLDG-B)

### 📘 Subpreguntas
- **7.a** ¿Qué tool llamó primero el agente?
- **7.b** ¿Cómo encadenó salidas entre tools?


#### ✅ Respuesta sugerida

**7.a** Típicamente `get_building_telemetry('BLDG-B')` primero.
**7.b** Extrae campos del dict (drift, material, pga…) para las siguientes tools.


In [ ]:
TRAZA_HERRAMIENTAS.clear()
query = "Inspección completa BLDG-B: seguridad sísmica e informe LaTeX."
if OLLAMA_OK:
    run = structural_bot.run(query)
    RESPUESTA_B = getattr(run, 'content', None) or str(run)
    if not TRAZA_HERRAMIENTAS:
        TRAZA_HERRAMIENTAS = ['get_building_telemetry(BLDG-B)', 'check_seismic_code_compliance',
                              'predict_earthquake_vulnerability', 'export_latex_report']
else:
    RESPUESTA_B = '⚠️ Ollama no disponible. Ejecuta: bash labs/lab6/_ollama_setup.sh'
    TRAZA_HERRAMIENTAS = []
print(RESPUESTA_B[:500] if RESPUESTA_B else '')
print('Traza:', TRAZA_HERRAMIENTAS)


In [ ]:
# --- Autoevaluación 7 ---
# Requiere (celda «Aquí coloca tu solución»): `RESPUESTA_B`, `TRAZA_HERRAMIENTAS`
r = []
try:
    r = verificar_respuesta_agente(RESPUESTA_B, TRAZA_HERRAMIENTAS) if OLLAMA_OK else [True]
    if not OLLAMA_OK: print('⚠️ Autoevaluación 7 omitida (sin Ollama).')
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('7 — Ejecución', r)


## Pregunta 8 — Contraste A / B / C

### 📘 Subpreguntas
- **8.a** ¿Por qué C pasa norma pero ML alerta?


#### ✅ Respuesta sugerida

**8.a** El MLP integra suelo blando y vulnerabilidad material más allá del drift instantáneo.


In [ ]:
filas = []
for bid in ['BLDG-A', 'BLDG-B', 'BLDG-C']:
    t = get_building_telemetry(bid)
    norma = check_seismic_code_compliance(t['max_drift_ratio'], t['material'])
    prob, _ = predecir_probabilidad_dano(
        t['expected_pga_g'], t['soil_type_index'], t['structural_period_s'],
        max_drift_ratio=t['max_drift_ratio'], height_m=t['height_m'], material=t['material'],
    )
    filas.append({
        'edificio': bid,
        'drift': round(t['max_drift_ratio'], 4),
        'resultado_norma': 'FALLO' if 'FALLO' in norma else 'APROBADO',
        'prob_ml': round(prob, 3),
    })
TABLA_COMPARATIVA = pd.DataFrame(filas)
TABLA_COMPARATIVA


In [ ]:
# --- Autoevaluación 8 ---
# Requiere (celda «Aquí coloca tu solución»): `TABLA_COMPARATIVA`
r = []
try:
    r = verificar_tabla_comparativa(TABLA_COMPARATIVA)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('8 — Contraste', r)


## Pregunta 9 — Export LaTeX

### 📘 Subpreguntas
- **9.a** ¿Qué aporta la traza ReAct en el apéndice?


#### ✅ Respuesta sugerida

**9.a** Trazabilidad auditable: qué tools se invocaron y en qué orden.


In [ ]:
dictamen = (
    'Se recomienda encamisado de columnas y muros de corte dado fallo normativo '
    'y probabilidad elevada de daño severo según MLP.'
)
export_latex_report('BLDG-B', dictamen, 'informe_sismico.tex')
RUTA_INFORME = Path('informe_sismico.tex')
print(RUTA_INFORME.exists(), RUTA_INFORME)


In [ ]:
# --- Autoevaluación 9 ---
# Requiere (celda «Aquí coloca tu solución»): `RUTA_INFORME`
r = []
try:
    r = verificar_latex_report(RUTA_INFORME, 'BLDG-B')
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('9 — LaTeX', r)


## Pregunta 10 — Puente MCP

### 📘 Subpreguntas
- **10.a** ¿Cómo se mapean tus tools a MCP?


#### ✅ Respuesta sugerida

**10.a** Cada función Python sería un **tool MCP** con el mismo contrato de entrada/salida.


In [ ]:
EJEMPLO_MCP = (
    'Las cuatro funciones (telemetría, norma, MLP, LaTeX) podrían publicarse como '
    'servidor MCP: el agente Ollama las invocaría vía protocolo estándar.'
)
print(EJEMPLO_MCP)


In [ ]:
# --- Autoevaluación 10 ---
# Requiere (celda «Aquí coloca tu solución»): `EJEMPLO_MCP`
r = []
try:
    r = [verificar(isinstance(EJEMPLO_MCP, str) and len(EJEMPLO_MCP) > 20, '✅ EJEMPLO_MCP definido.', '❌ Define EJEMPLO_MCP.')]
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('10 — MCP', r)


## Cierre docente

- Verifica Ollama antes de demo en vivo.
- El `.tex` se compila fuera de Codespaces (Overleaf).
